# 4.审核与编辑模式

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from typing import Literal
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.types import Command, interrupt
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model("deepseek-flash")


# 声明状态：MessagesState 自带 messages: Annotated[list[AnyMessage], add_messages]
# 用户输入/审核意见(HumanMessage)、模型稿件(AIMessage)统一追加进 messages，天然就是完整对话记录
class State(MessagesState):
    pass


# 声明节点
def llm_node(state: State) -> dict:
    resp = model.invoke(state["messages"])
    return {"messages": [resp]}


def review_node(state: State) -> Command[Literal["llm_node", END]]:
    review_msg = interrupt(
        {"instruction": "请审核大模型生成的内容，输入 y/yes 通过；或直接输入修改意见", }
    )
    if str(review_msg).strip().lower() in ("y", "yes"):
        return Command(goto=END)
    # 审核意见作为新的 HumanMessage 追加，goto 回 llm_node 按意见修改
    return Command(goto="llm_node", update={"messages": [HumanMessage(str(review_msg))]})


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("review_node", review_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "review_node")
builder.add_edge("review_node", END)

# 创建检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断：初始输入作为第一条消息
res = graph.invoke({"messages": [HumanMessage("写一首关于月亮的短诗，只需要提供短诗")]}, config=config)

# 人工审核循环：输入修改意见 -> 督促大模型按意见优化；输入 yes -> 通过结束
while res.get("__interrupt__"):
    user_in = input(res["__interrupt__"][0].value['instruction'])
    res = graph.invoke(Command(resume=user_in), config=config)

rprint(res)

In [ ]:
# 打印图结构
from IPython.display import display
display(graph)